# Bronze Ingestion: Event Hubs (Clickstream) -> Bronze (Structured Streaming)

Continuously reads clickstream events from Azure Event Hubs (via its
Kafka-compatible endpoint) and writes them into a Bronze Delta table
using Spark Structured Streaming.

**Key difference from the Postgres notebook:** this runs continuously
(a long-running streaming query), not a one-shot batch job. A checkpoint
location tracks exactly which events have been processed, so restarts
don't duplicate or lose data.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
)

## Configuration

In [0]:
SECRET_SCOPE = "retailflow_scope"

EVENTHUB_CONN_STR = dbutils.secrets.get(scope=SECRET_SCOPE, key="eventhub-connection-string")
EVENTHUB_NAME = dbutils.secrets.get(scope=SECRET_SCOPE, key="eventhub-name")

CATALOG = "retailflow"
BRONZE_TABLE = f"{CATALOG}.bronze.clickstream_events"
CHECKPOINT_PATH = f"/Volumes/{CATALOG}/landing/raw_files/_checkpoints/clickstream_events"

## Build the Kafka-compatible connection config

Event Hubs exposes a Kafka protocol endpoint on port 9093. Authentication
uses SASL with the connection string as the password, formatted as a
JAAS config string.

In [0]:
EVENTHUB_NAMESPACE_FQDN = EVENTHUB_CONN_STR.split(";")[0].replace("Endpoint=sb://", "").replace("/", "")
BOOTSTRAP_SERVERS = f"{EVENTHUB_NAMESPACE_FQDN}:9093"

EH_SASL = (
    "kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required "
    f'username="$ConnectionString" password="{EVENTHUB_CONN_STR}";'
)

KAFKA_OPTIONS = {
    "kafka.bootstrap.servers": BOOTSTRAP_SERVERS,
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.jaas.config": EH_SASL,
    "subscribe": EVENTHUB_NAME,
    "startingOffsets": "earliest",   # read from the beginning of the 24hr retention window on first run
    "failOnDataLoss": "false",       # tolerate retention-expired offsets rather than crashing
}

## Define the expected event schema

Matches the clickstream event shape produced by `produce_clickstream.py`.

In [0]:
event_schema = StructType([
    StructField("event_id", StringType()),
    StructField("event_type", StringType()),
    StructField("event_time", StringType()),   # parsed to timestamp below
    StructField("session_id", StringType()),
    StructField("customer_id", IntegerType()),
    StructField("device", StringType()),
    StructField("page_url", StringType()),
    StructField("referrer", StringType()),
    StructField("user_agent", StringType()),
    StructField("ip_address", StringType()),
    StructField("product_id", IntegerType()),
    StructField("quantity", IntegerType()),
    StructField("search_term", StringType()),
    StructField("cart_value", DoubleType()),
])

## Read the stream from Event Hubs

In [0]:
raw_stream_df = (
    spark.readStream.format("kafka")
    .options(**KAFKA_OPTIONS)
    .load()
)

## Parse the JSON payload

Kafka/Event Hubs messages arrive with the actual event as a raw `value`
column (bytes). We cast it to string, parse the JSON using our schema,
and add ingestion metadata -- same pattern as the Bronze Postgres notebook.

In [0]:
parsed_stream_df = (
    raw_stream_df
    .selectExpr("CAST(value AS STRING) as json_value", "timestamp as kafka_timestamp")
    .withColumn("data", F.from_json(F.col("json_value"), event_schema))
    .select("data.*", "kafka_timestamp")
    .withColumn(
        "event_time",
        F.coalesce(
            F.try_to_timestamp("event_time"),                              # try default ISO parsing first
            F.try_to_timestamp("event_time", F.lit("yyyy/MM/dd HH:mm:ss")), # fallback: slash format seen in the error
            F.try_to_timestamp("event_time", F.lit("yyyy-MM-dd'T'HH:mm:ss")) # fallback: ISO without offset
        )
    )
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_system", F.lit("eventhub_clickstream"))
)

## Write the stream to Bronze

`outputMode("append")` -- clickstream events are immutable facts, never
updated, so we only ever add new rows (no merge/upsert needed here,
unlike the Postgres OLTP tables).

The checkpoint location is what makes this restart-safe: Spark records
exactly which Kafka offsets have been processed, so re-running this cell
after a cluster restart resumes from where it left off instead of
reprocessing or skipping events.

In [0]:
query = (
    parsed_stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(availableNow=True)   # process what's available, then stop -- schedulable as a Job
    .toTable(BRONZE_TABLE)
)

query.awaitTermination()   # block until this run completes, so the notebook (and any Job running it) finishes cleanly

print(f"Run complete. Processed available events into {BRONZE_TABLE}.")

## Quick validation (run after the stream has processed a few batches)

In [0]:
display(spark.sql(f"""
    SELECT event_type, COUNT(*) as event_count
    FROM {BRONZE_TABLE}
    GROUP BY event_type
    ORDER BY event_count DESC
"""))